In [3]:
from pathlib import Path

import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---------- data path: ../data first, else search up the folder tree ----------
FILENAME = "plot4_public-investment.csv"
PATH = Path("../data") / FILENAME
if not PATH.exists():
    PATH = next(
        p for parent in [Path.cwd(), *list(Path.cwd().parents)[:4]]
        for p in parent.rglob(FILENAME)
    )
print("Using:", PATH.resolve())

# ---------- settings ----------
SOURCE = "Source: [add source]"                      # <- fill in
BARS, LINE = "Real gross public investment", "Deflator"
GREY = "#676A86"

# ---------- data ----------
raw = pd.read_csv(PATH, encoding="utf-8-sig")        # file has a BOM
raw = raw.rename(columns={"Year": "year"}).dropna(subset=[BARS, LINE], how="all")   # drops the empty 1991 row
raw["year"] = raw["year"].astype(int)

long = raw.melt(
    id_vars="year", value_vars=[BARS, LINE], var_name="series", value_name="value",
).dropna()

first, last = int(raw["year"].min()), int(raw["year"].max())
tick_years = [int(y) for y in raw["year"] if y % 5 == 0]

# ---------- chart ----------
x = alt.X("year:O", title=None, axis=alt.Axis(values=tick_years, labelAngle=0))
y = alt.Y("value:Q", title="Annual change", axis=alt.Axis(labelExpr="datum.label + '%'"))
color = alt.Color(
    "series:N", scale=alt.Scale(domain=[BARS, LINE]),     # bars take the theme's first colour, line the second
    legend=alt.Legend(
        orient="bottom", columns=2, symbolLimit=0,
        columnPadding=40, title=None, labelFontSize=10,
    ),
)

bars = alt.Chart(long[long["series"] == BARS]).mark_bar().encode(x=x, y=y, color=color)
zero = alt.Chart(pd.DataFrame({"value": [0]})).mark_rule(
    color=GREY, strokeWidth=0.8).encode(y=y)
line = alt.Chart(long[long["series"] == LINE]).mark_line(strokeWidth=1.8).encode(
    x=x, y=y, color=color)

caption = alt.Title(
    text=SOURCE,
    subtitle=[f"Real gross public investment (bars) and deflator (line), "
              f"annual change, %, {first}–{last}."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color=GREY, subtitleColor=GREY, dy=12)

chart = (
    (bars + zero + line)
    .properties(width=400, height=300)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor=GREY, titleColor=GREY))

styles.save(chart, path=".", name="germany_public_investment", svg=True)
chart.save("germany_public_investment.png", scale_factor=2.0)
chart

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Peterson Blogs/Roger Schneider/data/plot4_public-investment.csv


alt.LayerChart(...)